# Presenter walkthrough — `session1_part1_cpu.ipynb`

**Solved copy, for driving the session live.** It clones the `session1-part1-solution` branch, so every cell below runs as-is with no editing; attendees run the exercise notebook, which does not. The narration lives in the deck, `presentation/session1_part1.pdf`.

Generated by `presentation/generate_walkthrough.py` — do not edit this notebook; re-run the script.


# SC26 Tutorial — Session 1, Part 1: Introduction to MLIR (CPU)

You are going to compile and run a matrix-matrix product, `out[i, j] += sum_k a[i, k] * b[k, j]`, two ways:

1. Written by hand as `scf.for` loops, lowered to LLVM and JIT-executed.
2. Named by a single custom op, `blas.matmul`, and lowered to that same loop form by a compiler pass you write.

**You write the MLIR and the C++; the notebook cells are all provided.** Every edit happens in a source file in the repo, and the cells below check your work — so you never have to type code into Colab.

For each exercise there is a **solved matrix-*vector* example** that you can use as a reference.

Everything in this part runs on CPU only — no GPU runtime needed.

**Working locally instead (optional).** On Linux x86-64, you can run this notebook in VS Code:

1. Clone the repo: `git clone https://github.com/MaheshRavishankar/sc26_tutorial`
2. Download the toolchain, [`mlir-tutorial-llvm.tar.gz`](https://github.com/MaheshRavishankar/sc26_tutorial/releases/download/session1-toolchain-v1/mlir-tutorial-llvm.tar.gz), and untar it: `mkdir mlir-tutorial-llvm && tar xf mlir-tutorial-llvm.tar.gz -C mlir-tutorial-llvm --strip-components=1`
3. Before you start a kernel, create a file `.env` in the repo root with two lines, using absolute paths:
   ```
   MLIR_TUTORIAL_INSTALL_DIR=/path/to/mlir-tutorial-llvm
   MLIR_TUTORIAL_REPO_DIR=/path/to/sc26_tutorial
   ```
4. Open the repo folder in VS Code, then this notebook, and pick a Python 3.13 kernel with `ipykernel`, `numpy` and `requests` installed. Step 2 also needs `cmake`, `ninja` and a C++ compiler.

The setup cells then use your toolchain and your clone, and skip the download and the clone.

## Setup

**Run the cells in order, starting here.** Later cells use names defined by earlier ones, so skipping ahead can lead to strange errors. `Ctrl+F8` (run every cell above) catches you up if you lose your place.

The next cells download a prebuilt MLIR toolchain built for this tutorial: `mlir-opt`, `mlir-tblgen`, the MLIR Python bindings and CUDA runtime support. The [repo README](https://github.com/MaheshRavishankar/sc26_tutorial) explains why it is a custom build rather than the official LLVM release or a pip package.

In [ ]:
REPO = "MaheshRavishankar/sc26_tutorial"
RELEASE_TAG = "session1-toolchain-v1"
ASSET_NAME = "mlir-tutorial-llvm.tar.gz"

In [ ]:
import os
import requests

# Set MLIR_TUTORIAL_INSTALL_DIR to iterate locally (e.g. in VS Code) against
# an already-built toolchain -- skips the download entirely if it's already
# there. Unset in Colab, so this always defaults to the normal /content path.
INSTALL_DIR = os.environ.get("MLIR_TUTORIAL_INSTALL_DIR", "/content/mlir-tutorial-llvm")

if os.path.exists(f"{INSTALL_DIR}/bin/mlir-opt"):
    print(f"Using existing toolchain at {INSTALL_DIR}")
else:
    tar_path = f"{INSTALL_DIR}.tar.gz"
    url = f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(tar_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

    !mkdir -p {INSTALL_DIR}
    !tar xf {tar_path} -C {INSTALL_DIR} --strip-components=1

In [ ]:
import os, sys

os.environ["PATH"] = f"{INSTALL_DIR}/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{INSTALL_DIR}/python_packages/mlir_core")

!mlir-opt --version

In [ ]:
from mlir.ir import Context, Module
from mlir.passmanager import PassManager
from mlir.execution_engine import ExecutionEngine
import mlir.runtime as rt
import ctypes
import numpy as np

print("MLIR Python bindings imported OK")

The next cell clones the tutorial's repo.

In [ ]:
# Set MLIR_TUTORIAL_REPO_DIR to iterate locally against an already-cloned
# checkout -- skips the clone entirely if it's already there.
REPO_DIR = os.environ.get("MLIR_TUTORIAL_REPO_DIR", "/content/sc26_tutorial")
if os.path.isdir(REPO_DIR):
    print(f"Using existing repo checkout at {REPO_DIR}")
else:
    # --no-single-branch: every branch's tip, still at depth 1. It costs
    # nothing measurable, and it means the solution branches are already in
    # your checkout if you want a solved file out of one.
    WALKTHROUGH_BRANCH = "session1-part1-solution"
    !git clone --depth 1 --no-single-branch --branch {WALKTHROUGH_BRANCH} https://github.com/{REPO} {REPO_DIR}

## Step 1 — the loop nest, by hand

**Exercise 1: fill in the empty `@matmul` body of `session1/matmul_scf.mlir`** and save with Ctrl/Cmd+S. The cell below opens it in the editor, next to its reference.

**The reference is `session1/matvec_scf.mlir`,** a solved matrix-vector product. Matmul is the same loop nest with one more loop:

- an outer loop `%i` over the rows of `a`;
- inside it, `%j` over the columns of `b`;
- inside that, the same `iter_args` reduction as matvec, now over `%k`, with its result stored to `out[%i, %j]`.

**Like matvec, matmul adds to `out` (`+=`) instead of overwriting it.** So the reduction starts from `memref.load %out[%i, %j]`, not from zero.

The examples use fixed sizes (4x3 and so on) to keep them simple. The loops take their bounds from `memref.dim`, so they would work unchanged on sizes known only at run time (`memref<?x?xf32>`).

The four cells below are provided. Re-run them after each edit; they fail until the loop nest is right.

In [ ]:
# Opens the files below. On Colab, files.view() puts them in the editor
# pane -- use it instead of the file browser, which refuses some file
# types. Running locally there is no google.colab, so this prints the
# paths instead and you open them in your own editor.

try:
    from google.colab import files

    files.view(f"{REPO_DIR}/session1/matmul_scf.mlir")   # you write this
    files.view(f"{REPO_DIR}/session1/matvec_scf.mlir")   # reference, complete
except ModuleNotFoundError:
    print("open in your editor:")
    print(f"    {REPO_DIR}/session1/matmul_scf.mlir   (you write this)")
    print(f"    {REPO_DIR}/session1/matvec_scf.mlir   (reference, complete)")

First, parse and re-print the file with `mlir-opt`. No pass pipeline is given, so this just round-trips it through MLIR's parser and printer — the quickest way to catch a syntax error or a type mismatch in what you wrote.

In [ ]:
MATMUL_SCF_FILE = f"{REPO_DIR}/session1/matmul_scf.mlir"

!mlir-opt {MATMUL_SCF_FILE}
assert _exit_code == 0

Now parse it into an in-memory MLIR module on the Python side.

In [ ]:
ctx = Context()
module = Module.parseFile(MATMUL_SCF_FILE, ctx)
print(module)

Lower `scf` -> `cf` -> `llvm` in place. The cell prints the result, which is long, low-level LLVM-dialect IR: there is no need to read it. It is there to show that the loops are gone. `llvm-request-c-wrappers` makes `@matmul` callable from Python via `ExecutionEngine.invoke`.

In [ ]:
CPU_LOWERING_PIPELINE = (
    "builtin.module("
    "func.func(llvm-request-c-wrappers),"
    "convert-scf-to-cf,"
    "convert-cf-to-llvm,"
    "convert-arith-to-llvm,"
    "finalize-memref-to-llvm,"
    "convert-func-to-llvm,"
    "reconcile-unrealized-casts"
    ")"
)

PassManager.parse(CPU_LOWERING_PIPELINE, ctx).run(module.operation)
print(module)

JIT-compile the lowered module and invoke it with numpy arrays, checking the result against `numpy`'s own matmul.

Until `matmul_scf.mlir` is filled in, this fails on an all-zero result rather than an error — an empty function body is perfectly valid MLIR, it just never writes anything. That is what "not done yet" looks like here.

In [ ]:
def run_and_check(module, entry_point, args, expected, shared_libs=None, needs_init=False):
    """JIT-compile `module` (already lowered to LLVM) and call
    `entry_point(*args)`. Each arg is a numpy array passed as a memref; by
    convention the last one is the output buffer the kernel writes into. It is
    zeroed first -- we reuse the same buffers across steps, and without this a
    kernel that writes nothing would pass on the previous step's result -- then
    checked against `expected` and returned."""
    args[-1][...] = 0

    ee = ExecutionEngine(module, shared_libs=shared_libs or [])
    if needs_init:
        ee.initialize()
    ee.invoke(
        entry_point,
        *[ctypes.pointer(ctypes.pointer(rt.get_ranked_memref_descriptor(a)))
          for a in args],
    )

    out = args[-1]
    print("out      =", out)
    print("expected =", expected)
    np.testing.assert_allclose(out, expected)
    print("OK")
    return out


a = np.arange(12, dtype=np.float32).reshape(4, 3)
b = np.arange(15, dtype=np.float32).reshape(3, 5)
out = np.zeros((4, 5), dtype=np.float32)

run_and_check(module, "matmul", [a, b, out], expected=a @ b)

## Step 2 — the `blas` dialect

In this step you define a custom op, `blas.matmul`, and write the pass that lowers it to the loops from step 1. Both live in a dialect, `blas`, in `session1/blas-dialect/`. The dialect and its passes are compiled into one plugin, `libBlasPlugin.so`, which `mlir-opt` loads at run time with `--load-dialect-plugin` and `--load-pass-plugin`.

**First, build the plugin as it is in the repo.** It already defines `blas.matvec`, the reference for this step; `blas.matmul` is yours to add. The first build takes 1–2 minutes.

In [ ]:
import shutil

# Colab has neither; skip the install where both are already on PATH (and
# where apt-get would need root, e.g. running locally).
if shutil.which("cmake") and shutil.which("ninja"):
    print("cmake and ninja are already installed")
else:
    !apt-get install -y -qq cmake ninja-build > /dev/null
    assert _exit_code == 0

In [ ]:
PLUGIN_SRC = f"{REPO_DIR}/session1/blas-dialect"
PLUGIN_BUILD = f"{PLUGIN_SRC}/build"
PLUGIN_SO = f"{PLUGIN_BUILD}/libBlasPlugin.so"

# Configure and build the plugin as it is in the repo. After you edit a
# .td.txt or .cpp file, the rebuild cell further down is enough.
!cmake -S {PLUGIN_SRC} -B {PLUGIN_BUILD} -G Ninja \
    -DMLIR_DIR={INSTALL_DIR}/lib/cmake/mlir \
    -DLLVM_DIR={INSTALL_DIR}/lib/cmake/llvm \
    -DCMAKE_BUILD_TYPE=Release
assert _exit_code == 0
!ninja -C {PLUGIN_BUILD}
assert _exit_code == 0

Round-trip `session1/matvec_custom.mlir` through the plugin. It uses `blas.matvec`, which the plugin defines. Python's `Context` does not know about the plugin, so anything involving `blas` goes through `mlir-opt` as a subprocess. With no pass pipeline, `mlir-opt` just parses the file and prints it back, using the parser and printer generated from the op's ODS.

`blas.matvec` comes back in the same syntax as in the file. (`mlir-opt` names the arguments `%arg0`, `%arg1` and `%arg2`, and wraps the function in a `module`.) That is what the check on your own op will look like.

In [ ]:
MATVEC_CUSTOM_FILE = f"{REPO_DIR}/session1/matvec_custom.mlir"

!mlir-opt --load-dialect-plugin={PLUGIN_SO} {MATVEC_CUSTOM_FILE}
assert _exit_code == 0

**Exercise 2: define a custom op, `blas.matmul`, that represents the loop nest you wrote in step 1 as a single operation.** `session1/matmul_custom.mlir` uses it, and the next cell prints that file. Your definition is right when `mlir-opt` reads the file and prints `blas.matmul` back in the same syntax. In Exercise 4, you write the pass that lowers it back to the loops.

Ops are defined in ODS (Operation Definition Specification), a TableGen file describing an op's operands, types and syntax. The build runs `mlir-tblgen` on it to generate the op's C++ class. In `session1/blas-dialect/include/Blas/BlasOps.td.txt`, define `Blas_MatMulOp` where the `TODO` is, including `let hasVerifier = 1;` as `Blas_MatVecOp` does. (TableGen files are normally `.td`, but Colab's editor will not open that extension, so this one is `.td.txt`.)

**Exercise 3: write the op's verifier.** ODS checks the operand types, but it cannot relate their shapes to each other. So `hasVerifier = 1` declares a `LogicalResult MatMulOp::verify()`, which you write in `session1/blas-dialect/lib/BlasOps.cpp`. `blas.matmul` has three constraints: the reduction dimension, the rows and the columns. Each one is a single call to the `verifyDimsMatch` helper in that file. The helper lets a `?` dimension through, because its size is only known at run time.

**The references are the `matvec` versions, directly above each `TODO`.**

**Do both before you rebuild.** Declaring `hasVerifier = 1` without writing `verify()` still builds, because the plugin is linked with `-Wl,-z,undefs` (`mlir-opt` resolves MLIR's own symbols when it loads it). It then fails to load:

```
symbol lookup error: libBlasPlugin.so: undefined symbol: _ZN4blas8MatMulOp6verifyEv
```

which demangles to `blas::MatMulOp::verify()`.

**Then run the cells below one at a time.** They show the op you are defining, open the two files in the editor, rebuild the plugin and check your op. After each edit, re-run the rebuild cell and the check after it.

In [ ]:
MATMUL_CUSTOM_FILE = f"{REPO_DIR}/session1/matmul_custom.mlir"

!cat {MATMUL_CUSTOM_FILE}

In [ ]:
# Opens the files below. On Colab, files.view() puts them in the editor
# pane -- use it instead of the file browser, which refuses some file
# types. Running locally there is no google.colab, so this prints the
# paths instead and you open them in your own editor.

BLAS_DIALECT = f"{REPO_DIR}/session1/blas-dialect"

try:
    from google.colab import files

    files.view(f"{BLAS_DIALECT}/include/Blas/BlasOps.td.txt")   # define the op
    files.view(f"{BLAS_DIALECT}/lib/BlasOps.cpp")   # write its verifier
except ModuleNotFoundError:
    print("open in your editor:")
    print(f"    {BLAS_DIALECT}/include/Blas/BlasOps.td.txt   (define the op)")
    print(f"    {BLAS_DIALECT}/lib/BlasOps.cpp   (write its verifier)")

In [ ]:
# Rebuild the plugin. Re-run this cell after EVERY edit to a .td.txt or .cpp
# file -- those are compiled into libBlasPlugin.so, unlike the .mlir files,
# which are read fresh each time. The assert matters: a failed build leaves
# the previous plugin in place, so the check below would keep testing your
# last working version.
!ninja -C {PLUGIN_BUILD}
assert _exit_code == 0

Round-trip `matmul_custom.mlir` through the plugin, as you did for matvec. **This is the check on Exercises 2 and 3:** your definition is right when `blas.matmul` comes back in the same syntax. `custom op 'blas.matmul' is unknown` means the op isn't defined yet, or the plugin didn't rebuild.

In [ ]:
!mlir-opt --load-dialect-plugin={PLUGIN_SO} {MATMUL_CUSTOM_FILE}
assert _exit_code == 0

The build ran `mlir-tblgen` on your `.td.txt` and turned it into C++, and that is where the names the next step uses come from: `getA()`, `getB()`, `getOut()` for the operands, the `build()` overloads a pattern calls to create the op, and the `verify()` you implemented. Nothing to edit here — it is regenerated on every build.

In [ ]:
import re

BLAS_OPS_HEADER = f"{PLUGIN_BUILD}/include/Blas/BlasOps.h.inc"
declaration = re.search(r"class MatMulOp :.*?\n\};", open(BLAS_OPS_HEADER).read(), re.S)
assert declaration, (
    "no MatMulOp in the generated header -- is the op defined in BlasOps.td.txt "
    "and the plugin rebuilt?")

print(f"{BLAS_OPS_HEADER}\n")
for line in declaration.group(0).split("\n"):
    if re.search(r"^class MatMulOp :|get[A-Z]\w*\(\)|static void build|LogicalResult verify\(", line):
        print(line.strip())

### Write the lowering

**Exercise 4: write the pattern that lowers `blas.matmul` to loops, in `session1/blas-dialect/lib/LowerBlasToSCF.cpp`.** One pass, `convert-blas-to-scf`, lowers the whole dialect, with one pattern per op. `MatVecToSCFPattern` is complete, and `MatMulToSCFPattern` is already there with its scaffolding commented out — the operands, the index constants, the problem sizes and the `eraseOp` that retires the op. Uncomment that, write the loop nest where it says so, and then **register the pattern** in `runOnOperation()`. The file has a `TODO` at each spot.

**How the pass finds your pattern.** The pass is declared in TableGen, like the op: `include/Blas/Passes.td` has `def LowerBlasToSCFPass : Pass<"convert-blas-to-scf", ...>`. That string is the name `mlir-opt` knows the pass by. Nothing is inferred from the C++ class name. `mlir-tblgen` generates the base class `LowerBlasToSCFPassBase`, which the pass in `LowerBlasToSCF.cpp` derives from. The plugin registers the pass with `mlir-opt` when it is loaded (`PluginEntry.cpp`). Inside the pass, `runOnOperation()` collects patterns with `patterns.add<...>` and hands them to `applyPatternsGreedily`, which tries each pattern on the ops it was written for. `OpRewritePattern<MatMulOp>` is what ties `MatMulToSCFPattern` to `blas.matmul`, so the pattern does nothing until you add it. More: [declaring passes](https://mlir.llvm.org/docs/PassManagement/#declarative-pass-specification), [patterns and the rewrite driver](https://mlir.llvm.org/docs/PatternRewriter/).

Until the pattern rewrites anything it returns `failure()`, so `blas.matmul` survives the pass — which is the same symptom as forgetting to register it.

**The reference is `MatVecToSCFPattern`, in the same file.**

Loop bodies there are built by moving the rewriter's insertion point into the loop's block, rather than by passing a body-builder lambda. One rule to know: a loop with no `iter_args` gets its `scf.yield` created for you, and a loop *with* `iter_args` does not — only you know what it should yield — so each reduction loop ends with an explicit `scf::YieldOp::create`.

Forgetting to register the pattern is caught for you: the pass reports which op it failed to convert, rather than leaving `blas.matmul` in the output to fail later as an unrelated-looking `Dialect 'blas' not found`.

**Then run the cells below one at a time.** They open the file, rebuild the plugin, run your pass and check the result. After each edit, re-run them from the rebuild cell on.

In [ ]:
# Opens the files below. On Colab, files.view() puts them in the editor
# pane -- use it instead of the file browser, which refuses some file
# types. Running locally there is no google.colab, so this prints the
# paths instead and you open them in your own editor.

try:
    from google.colab import files

    files.view(f"{BLAS_DIALECT}/lib/LowerBlasToSCF.cpp")   # pattern + registration
except ModuleNotFoundError:
    print("open in your editor:")
    print(f"    {BLAS_DIALECT}/lib/LowerBlasToSCF.cpp   (pattern + registration)")

In [ ]:
# Rebuild the plugin. Re-run this cell after EVERY edit to a .td.txt or .cpp
# file -- those are compiled into libBlasPlugin.so, unlike the .mlir files,
# which are read fresh each time. The assert matters: a failed build leaves
# the previous plugin in place, so the checks below would keep testing your
# last working version.
!ninja -C {PLUGIN_BUILD}
assert _exit_code == 0

Run your pass, writing the result to a file.

In [ ]:
LOWERED_MATMUL_FILE = "matmul_custom_lowered.mlir"

!mlir-opt --load-dialect-plugin={PLUGIN_SO} --load-pass-plugin={PLUGIN_SO} "--pass-pipeline=builtin.module(convert-blas-to-scf)" {MATMUL_CUSTOM_FILE} -o {LOWERED_MATMUL_FILE}
assert _exit_code == 0
!cat {LOWERED_MATMUL_FILE}

Load the lowered file, run it through the same CPU pipeline as step 1, JIT it, and check the result. Compare the output above with your `matmul_scf.mlir`: the pass built the same thing you wrote by hand.

In [ ]:
custom_module = Module.parseFile(LOWERED_MATMUL_FILE, ctx)
PassManager.parse(CPU_LOWERING_PIPELINE, ctx).run(custom_module.operation)
run_and_check(custom_module, "matmul", [a, b, out], expected=a @ b)